# Partitioning e compaction strategies

In [ ]:
import os
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *

MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppClass03") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/tmp/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/tmp/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

In [ ]:
spark

## PROJECT 1: HOTEL BOOKING

- [Data source](https://www.kaggle.com/datasets/mojtaba142/hotel-booking)

## READING STAGING (RAW)

In [ ]:
location_raw = f"s3a://staging"
file = "hotel_booking.csv"
data_origen = f"{location_raw}/{file}"

In [ ]:
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(data_origen)
df = df.withColumnRenamed("phone-number", "phone_number")

In [ ]:
df.createOrReplaceTempView("staging")

## Choice the partition column

1. Is the cardinality of a column very high?

    If so, do not use that column for partitioning. For example, if you partition by a
    column userId and there can be more than a million distinct user IDs, then that
    is a bad partitioning strategy.

2. Is the balanced distribution?

    The data should be distributed evenly among partitions to prevent some from becoming too large while others remain nearly empty.

3. How often are filters?

    The column should be commonly used in WHERE clauses in queries to leverage the benefits of partitioning and enhance performance.

4. How much data will exist in each partition?

    You can partition by a column if you expect data in that partition to be at least
    1 GB.

In [ ]:
spark.sql(f"""
    SELECT
      COUNT(DISTINCT name) AS unique_name_values,
      COUNT(DISTINCT email) AS unique_email_values,
      COUNT(DISTINCT credit_card) AS unique_credit_card_values,
      COUNT(DISTINCT hotel) AS unique_hotel_values,
      COUNT(DISTINCT is_canceled) AS unique_is_canceled_values,
      COUNT(DISTINCT lead_time) AS unique_lead_time_values,
      COUNT(DISTINCT arrival_date_year) AS unique_arrival_date_year_values,
      COUNT(DISTINCT arrival_date_month) AS unique_arrival_date_month_values,
      COUNT(DISTINCT arrival_date_week_number) AS unique_arrival_date_week_number_values,
      COUNT(DISTINCT arrival_date_day_of_month) AS unique_arrival_date_day_of_month_values,
      COUNT(DISTINCT stays_in_weekend_nights) AS unique_stays_in_weekend_nights_values,
      COUNT(DISTINCT stays_in_week_nights) AS unique_stays_in_week_nights_values,
      COUNT(DISTINCT adults) AS unique_adults_values,
      COUNT(DISTINCT children) AS unique_children_values,
      COUNT(DISTINCT babies) AS unique_babies_values,
      COUNT(DISTINCT meal) AS unique_meal_values,
      COUNT(DISTINCT country) AS unique_country_values,
      COUNT(DISTINCT market_segment) AS unique_market_segment_values,
      COUNT(DISTINCT distribution_channel) AS unique_distribution_channel_values,
      COUNT(DISTINCT is_repeated_guest) AS unique_is_repeated_guest_values,
      COUNT(DISTINCT previous_cancellations) AS unique_previous_cancellations_values,
      COUNT(DISTINCT previous_bookings_not_canceled) AS unique_previous_bookings_not_canceled_values,
      COUNT(DISTINCT reserved_room_type) AS unique_reserved_room_type_values,
      COUNT(DISTINCT assigned_room_type) AS unique_assigned_room_type_values,
      COUNT(DISTINCT booking_changes) AS unique_booking_changes_values,
      COUNT(DISTINCT deposit_type) AS unique_deposit_type_values,
      COUNT(DISTINCT agent) AS unique_agent_values,
      COUNT(DISTINCT company) AS unique_company_values,
      COUNT(DISTINCT days_in_waiting_list) AS unique_days_in_waiting_list_values,
      COUNT(DISTINCT customer_type) AS unique_customer_type_values,
      COUNT(DISTINCT adr) AS unique_adr_values,
      COUNT(DISTINCT required_car_parking_spaces) AS unique_required_car_parking_spaces_values,
      COUNT(DISTINCT total_of_special_requests) AS unique_total_of_special_requests_values,
      COUNT(DISTINCT reservation_status) AS unique_reservation_status_values,
      COUNT(DISTINCT reservation_status_date) AS unique_reservation_status_date_values
    FROM staging;
""").show(truncate=False)

In [ ]:
spark.sql("SELECT * FROM staging LIMIT 20").show(truncate=False)

### 3 - Write data into partitioned bronze

In [ ]:
table_bronze_1 = "hotel_booking_partitioned_bronze"
table_bronze_2 = "hotel_booking_partitioned_bronze_2"
location_bronze_1 = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze_1}"
location_bronze_2 = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze_2}"

### Let's write using a bad partitioning strategy

In [ ]:
%%time
# Writing in Delta format
(
    df.write.format("delta")
    .mode("overwrite")
    .partitionBy("reservation_status_date")
    .save(location_bronze)
)

### Now, let's write using a good partitioning strategy

In [ ]:
%%time
(
    df.write.format("delta")
    .mode("overwrite")
    .partitionBy("arrival_date_year", "arrival_date_month")
    .save(location_bronze)
)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze_1}
    USING DELTA
    LOCATION '{location_bronze_1}'
""")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze_2}
    USING DELTA
    LOCATION '{location_bronze_2}'
""")

In [ ]:
%%time
spark.sql(f"SELECT COUNT(*) FROM {DATABASE}.{table_bronze_1} WHERE reservation_status_date = '2015-12-15'").show()

In [ ]:
%%time
spark.sql(f"""
SELECT COUNT(*) 
FROM {DATABASE}.{table_bronze_2} 
WHERE 
    arrival_date_year = '2015'
    AND arrival_date_month = 'August'
""").show(truncate=False)

In [ ]:
spark.sql(f"""
DESCRIBE DETAIL {DATABASE}.{table_bronze_1} 
""").show(truncate=False)

In [ ]:
spark.sql(f"""
DESCRIBE DETAIL {DATABASE}.{table_bronze_2} 
""").show(truncate=False)

In [ ]:
spark.stop()